# Day 51: Capstone Explainable AI
Using SHAP and Feature Importance to explain our model's predictions to business stakeholders.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import shap

# Initialize JS visualization for SHAP
shap.initjs()

# Load Data
df = pd.read_csv('../day49-capstone-baseline/customer_data.csv')
print(df.head())

## Train the Tuned Model
We use the optimal hyperparameters discovered in Day 50.

In [ ]:
X = pd.get_dummies(df.drop(columns=['customer_id', 'churn']), drop_first=True)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

best_rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=2, random_state=42)
best_rf.fit(X_train, y_train)
print('Model trained successfully.')

## Global Explainability: Feature Importance
Which features are most important across all customers?

In [ ]:
importances = best_rf.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title('Feature Importances')
plt.bar(range(X.shape[1]), importances[indices], align='center')
plt.xticks(range(X.shape[1]), feature_names[indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Advanced Explainability: SHAP Values
SHAP (SHapley Additive exPlanations) provides a unified measure of feature importance and reveals the direction of the impact.

In [ ]:
explainer = shap.TreeExplainer(best_rf)
# Sample to speed up computation
X_test_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

# Depending on SHAP version, shap_values might be a list
if isinstance(shap_values, list):
    shap_to_plot = shap_values[1]
else:
    shap_to_plot = shap_values

shap.summary_plot(shap_to_plot, X_test_sample)

## Local Explainability: Explaining a Single Prediction
Let's see why a specific customer was predicted to churn.

In [ ]:
# Explain the first customer in our test sample
# shap.force_plot(explainer.expected_value[1], shap_to_plot[0,:], X_test_sample.iloc[0,:])
# For newer shap versions:
shap.plots.waterfall(shap.Explanation(values=shap_to_plot[0], base_values=explainer.expected_value[1], data=X_test_sample.iloc[0,:], feature_names=X_test_sample.columns))